# Drift Monitoring

A model is validated against the inputs you tested on. In production the input
distribution moves -- new user segments, seasonal text, a changed upstream
feature. Drift monitoring is a leading indicator: it warns you the inputs
shifted before accuracy quietly degrades. This runs with no LLM calls.

## Reference window vs detection window

You compare a frozen reference window (what the model was validated on) against
a recent detection window (what it is seeing now). Numeric features use the
KS statistic; categorical features use PSI and Chi-square. A `FeatureSpec`
declares which tests apply and the alert thresholds.

In [ ]:
import sys
sys.path.insert(0, "../src")

from genai_prod_kit.drift.config import FeatureSpec, FeatureType, TestType
from genai_prod_kit.drift.monitor import monitor_numeric, monitor_categorical

numeric_spec = FeatureSpec(
    feature="input_length", feature_field="chars",
    feature_type=FeatureType.NUMERIC, tests=(TestType.KS,),
)
categorical_spec = FeatureSpec(
    feature="sentiment_label", feature_field="dist",
    feature_type=FeatureType.CATEGORICAL,
    tests=(TestType.PSI, TestType.CHI_SQUARE),
)
print("specs ready")

## Numeric drift (KS)

Same distribution -> KS statistic ~0 -> OK. A fully separated distribution ->
KS = 1.0 -> ALERT. The severity is derived from the KS critical values at
alpha=0.05 / 0.01, scaled by sample size.

In [ ]:
same = monitor_numeric(numeric_spec, list(range(50)), list(range(50)))
drift = monitor_numeric(numeric_spec, list(range(50)), list(range(1000, 1050)))
print("same  :", same)
print("drift :", drift)

## Categorical drift (PSI / Chi-square)

PSI quantifies how far the label mix moved; a large shift raises it past the
ALERT threshold. Chi-square is recorded as a statistic alongside it.

In [ ]:
ref = {"pos": 60, "neu": 30, "neg": 10}
same = monitor_categorical(categorical_spec, ref, dict(ref))
shift = monitor_categorical(categorical_spec, ref, {"pos": 10, "neu": 20, "neg": 70})
print("same  :", same)
print("shift :", shift)

## Sample-size guard (shadow by default)

Below `min_sample_size` (default 30) the monitor returns no result rather than
firing on noise. The whole monitor runs in shadow by default: it records and
warns, it does not block traffic.

In [ ]:
print("n=29 (below min):", monitor_numeric(numeric_spec, list(range(29)), list(range(29))))
print("n=30 (at min)   :", monitor_numeric(numeric_spec, list(range(30)), list(range(30))))
print("one side short  :", monitor_numeric(numeric_spec, list(range(50)), list(range(10))))

## Why a leading indicator matters

Accuracy regressions show up after users are already affected and only if you
have labels. Input drift is visible immediately, with no labels, from the
request stream alone -- so it buys you time to investigate before quality
actually drops.